# Part  3: Modeling

In [1]:
import joblib 
import numpy as np

X_train_scaled = joblib.load('../data/processed/X_train_scaled.pkl')
X_test_scaled = joblib.load('../data/processed/X_test_scaled.pkl')

y_train = joblib.load('../data/processed/y_train.pkl')  
y_test = joblib.load('../data/processed/y_test.pkl')

print(X_train_scaled.shape)
print(X_test_scaled.shape)
print(y_train.shape)
print(y_test.shape)

(1166, 234)
(292, 234)
(1166,)
(292,)


In [11]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_test_actual = np.expm1(y_test)

def evaluate_model_real_price(model, X_test, model_name):
    # predict log price
    y_pred_log = model.predict(X_test)
    
    # inverse log price to get actual price
    y_pred_actual = np.expm1(y_pred_log)
    
    # caculate evalation metrics
    rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
    mae = mean_absolute_error(y_test_actual, y_pred_actual)
    r2 = r2_score(y_test_actual, y_pred_actual)
    
    print(f"{model_name}")
    print(f"RMSE : {rmse:,.2f}")
    print(f"MAE  : {mae:,.2f}")
    print(f"R²   : {r2:.4f}\n")


***3.1 Linear Regression***

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np


lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train) # learn by OLS - thuật toán bình phương tối thiểu 
y_pred_lr = lr_model.predict(X_test_scaled)

evaluate_model_real_price(lr_model, X_test_scaled, "Linear Regression")

Linear Regression
RMSE : 22,022.45
MAE  : 15,559.12
R²   : 0.9122



**Linear Regression achieved highly accurate predictions.** it performed exceptionally well on this dataset for two reasons:

- The log-transformation of SalePrice corrected the right-skewed distribution, making relationships more linear.

- The model efficiently handled the sparse matrix generated by One-Hot Encoding.


***3.2 Random forest***

In [4]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)#n_estiomators: the number of decision trees
rf_model.fit(X_train_scaled, y_train) 

y_pred_rf = rf_model.predict(X_test_scaled)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

evaluate_model_real_price(rf_model, X_test_scaled, "Random Forest")


Random Forest
RMSE : 24,057.09
MAE  : 16,551.57
R²   : 0.8952



**Random Forest underperforms Linear Regression across all metrics (higher RMSE & MAE, lower R²)**


- Sparse Data: Our One-Hot Encoding generated over 200 binary features (0s and 1s). Tree-based models often find it difficult to make effective splits on such sparse matrices compared to linear models

- Lack of Tuning: The Random Forest was trained using default hyperparameters. Without limiting tree depth or adjusting leaf nodes, it is likely slightly overfitting the training data, leading to a drop in accuracy on the test set

***3.3 XGBoost-eXtreme Gradient Boosting***

In [5]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(n_estimators = 100, random_state = 42)
xgb_model.fit(X_train_scaled, y_train)

y_pred_xgb = xgb_model.predict(X_test_scaled)

evaluate_model_real_price(xgb_model, X_test_scaled, "XGBoost")

XGBoost
RMSE : 23,623.19
MAE  : 16,524.22
R²   : 0.8990



In [6]:
X_train = joblib.load('../data/processed/X_train.pkl')
import pandas as pd
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance.head(15))

                 feature  importance
4            OverallQual    0.203911
47               TotalSF    0.125938
191         CentralAir_Y    0.122834
33            GarageCars    0.061504
206    GarageType_Detchd    0.043214
36            GarageCond    0.037404
27          KitchenAbvGr    0.034064
9              ExterQual    0.026959
134  Exterior1st_BrkComm    0.017801
158   Exterior2nd_Stucco    0.017681
70         LandSlope_Mod    0.017225
170      BsmtExposure_Gd    0.014288
31           FireplaceQu    0.013671
54           MSZoning_RL    0.013481
28           KitchenQual    0.012765


***GridSearchCV***

In [7]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],#default is 6
    'learning_rate': [0.05, 0.1, 0.2]#default is 0.1
}

grid_search = GridSearchCV(
    XGBRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print("Best params:", grid_search.best_params_)
print("Best RMSE:", np.sqrt(-grid_search.best_score_))

Best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300}
Best RMSE: 0.12372372029340531


In [8]:
best_xgb_model = grid_search.best_estimator_

y_pred_best = best_xgb_model.predict(X_test_scaled)

rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best))
mae_best = mean_absolute_error(y_test, y_pred_best)
r2_best = r2_score(y_test, y_pred_best)

evaluate_model_real_price(best_xgb_model, X_test_scaled, "best_XGBoost")

best_XGBoost
RMSE : 19,650.11
MAE  : 13,787.39
R²   : 0.9301



**XGBoost**

The Tuned XGBoost (`best_XGBoost`) significantly outperforms all other baseline models, achieving the highest accuracy (R² = 0.9301) and the lowest error rates on real USD values (RMSE ≈ 19,650, MAE ≈ 13,787).

⇒ Therefore, the tuned XGBoost is  selected as the final model for deployment and future predictions.

In [9]:
import joblib

joblib.dump(best_xgb_model, '../model/house_price_model.pkl')

print("File saved successfully.")

File saved successfully.


In [10]:
#Check

loaded_model = joblib.load('../model/house_price_model.pkl')
loaded_scaler = joblib.load('../model/scaler.pkl')
loaded_columns = joblib.load('../model/feature_columns.pkl')

print("Số lượng feature columns:", len(loaded_columns))

test_sample = X_test_scaled[0].reshape(1, -1)
prediction = loaded_model.predict(test_sample)
performance = (1 - abs(prediction[0] - y_test.iloc[0]) / y_test.iloc[0]) * 100

print("Predictive Price:", prediction[0])
print("Actual Price:", y_test.iloc[0])
print(f"The performance for the first house in dataset: {performance:.2f}%")

Số lượng feature columns: 234
Predictive Price: 12.257436
Actual Price: 12.154784614286667
The performance for the first house in dataset: 99.16%


## Kết luận từ Modeling (Bước 1.3) - Khi chưa (Log-transform target + create new features)
<b>

1. So sánh 3 model trên test set: Linear Regression (R² 0.859) < Random Forest (R² 0.891) < XGBoost mặc định (R² 0.901)

2. GridSearchCV fit XGBoost → best params: learning_rate=0.05, max_depth=3, n_estimators=300

3. Final model: XGBoost was fixed — RMSE 20,232.65 / MAE 14,772.80 / R² 0.9259

4. Feature importance khớp phần lớn với correlation (OverallQual), nhưng GrLivArea tụt hạng do multicollinearity; các cột one-hot (GarageType_Detchd, CentralAir_Y) chỉ XGBoost phát hiện được

<b>


## Kết luận từ Modeling - (Log-transform target + create new features)
1. So sánh 3 model trên test set: Random Forest (R² 0.895) < XGBoost mặc định (R² 0.899) < Linear Regression (R² 0.912). 

2. GridSearchCV fit XGBoost → best params: learning_rate: 0.1, max_depth: 3, 'n_estimators': 300

3. Final model: best_XGBoost was fixed — RMSE 19,650.11 / MAE 13,787.39 / R² 0.9301.
